In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("OlistProject").master("local[*]").getOrCreate()

sc = spark.sparkContext

2026-09-03 07:37:01,929 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
2026-09-03 07:37:01,940 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-03 07:37:04,113 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# EDA and Preprocessing

In [2]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

orders = (
    spark.read
    .option("sep", ",")
    .schema(orders_schema)
    .csv("/user/hadoop/commerce_storage/orders")
)

orders.printSchema()
orders.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



+--------------------------------+--------------------------------+------------+------------------------+---------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at    |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+---------------------+----------------------------+-----------------------------+-----------------------------+
|00010242fe8c5a6d1ba2dd792cb16214|3ce436f183e68e07877b285a838db11a|delivered   |2017-09-13 08:59:02.0   |2017-09-13 09:45:35.0|2017-09-19 18:34:16.0       |2017-09-20 23:43:48.0        |2017-09-29 00:00:00.0        |
|00018f77f2f0320c557190d7a144bdd3|f6dd3ec061db4e3987629fe6b26e5cce|delivered   |2017-04-26 10:53:06.0   |2017-04-26 11:05:13.0|2017-

In [3]:
print("Number of rows:", orders.count())
print("Number of columns:", len(orders.columns))
print("Columns:", orders.columns)

null_report = orders.select([
    count(
        when(
            col(c).isNull() | (col(c) == ""),
            c
        )
    ).alias(c)
    for c in orders.columns
])

print("Null values per column:")
null_report.show(truncate=False)

print("Duplicate order_id count:")
duplicate_order_ids = (
    orders.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
print(duplicate_order_ids)

Number of rows: 99441
Number of columns: 8
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Null values per column:


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |0                |0                           |0                            |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

Duplicate order_id count:


0


In [4]:
print("Order status distribution:")
orders.groupBy("order_status").count().orderBy(col("count").desc()).show()

print("Zero delivery dates:")
orders.select(
    (col("order_delivered_carrier_date").startswith("0000-00-00")).alias("carrier_zero"),
    (col("order_delivered_customer_date").startswith("0000-00-00")).alias("customer_zero")
).groupBy("carrier_zero", "customer_zero").count().show()

print("Examples of missing/zero delivery dates:")
orders.filter(
    col("order_delivered_carrier_date").startswith("0000-00-00") |
    col("order_delivered_customer_date").startswith("0000-00-00") |
    col("order_delivered_carrier_date").isNull() |
    col("order_delivered_customer_date").isNull()
).select(
    "order_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)

Order status distribution:


+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+

Zero delivery dates:
+------------+-------------+-----+
|carrier_zero|customer_zero|count|
+------------+-------------+-----+
|       false|        false|99441|
+------------+-------------+-----+

Examples of missing/zero delivery dates:


+--------+------------+----------------------------+-----------------------------+
|order_id|order_status|order_delivered_carrier_date|order_delivered_customer_date|
+--------+------------+----------------------------+-----------------------------+
+--------+------------+----------------------------+-----------------------------+



In [5]:
print("Rows containing Hive null marker \\N:")
orders.filter(
    col("order_delivered_carrier_date").contains("N") |
    col("order_delivered_customer_date").contains("N")
).select(
    "order_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)

print("Distinct delivery values for non-delivered orders:")
orders.filter(col("order_status") != "delivered").select(
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).distinct().show(30, truncate=False)

Rows containing Hive null marker \N:
+--------+------------+----------------------------+-----------------------------+
|order_id|order_status|order_delivered_carrier_date|order_delivered_customer_date|
+--------+------------+----------------------------+-----------------------------+
+--------+------------+----------------------------+-----------------------------+

Distinct delivery values for non-delivered orders:


+------------+----------------------------+-----------------------------+
|order_status|order_delivered_carrier_date|order_delivered_customer_date|
+------------+----------------------------+-----------------------------+
|shipped     |2017-01-25 17:10:19.0       |null                         |
|shipped     |2017-02-22 09:00:29.0       |null                         |
|shipped     |2018-01-04 21:41:39.0       |null                         |
|shipped     |2017-05-24 12:31:48.0       |null                         |
|shipped     |2018-04-13 21:48:36.0       |null                         |
|shipped     |2017-10-13 18:27:28.0       |null                         |
|canceled    |2018-02-06 14:54:44.0       |null                         |
|shipped     |2017-12-13 01:19:11.0       |null                         |
|shipped     |2017-04-27 15:45:30.0       |null                         |
|shipped     |2017-12-06 20:32:56.0       |null                         |
|shipped     |2017-04-17 09:05:56.0   

In [6]:
null_report = orders.select([
    sum(
        when(
            col(c).isNull() | (col(c) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in orders.columns
])

null_report.show(truncate=False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |0                |0                           |0                            |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [7]:
orders.select(
    sum(when(col("order_delivered_carrier_date").isNull(), 1).otherwise(0)).alias("carrier_date_nulls"),
    sum(when(col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("customer_date_nulls"),
    sum(when(col("order_approved_at").isNull(), 1).otherwise(0)).alias("approved_date_nulls")
).show()

+------------------+-------------------+-------------------+
|carrier_date_nulls|customer_date_nulls|approved_date_nulls|
+------------------+-------------------+-------------------+
|                 0|                  0|                  0|
+------------------+-------------------+-------------------+



In [8]:
print("Literal null text count:")
orders.select(
    (col("order_delivered_carrier_date") == "null").cast("int").alias("carrier_null_text"),
    (col("order_delivered_customer_date") == "null").cast("int").alias("customer_null_text")
).groupBy().sum().show()

print("Empty text count:")
orders.select(
    (col("order_delivered_carrier_date") == "").cast("int").alias("carrier_empty"),
    (col("order_delivered_customer_date") == "").cast("int").alias("customer_empty")
).groupBy().sum().show()

print("Distinct values for missing customer delivery date:")
orders.filter(
    col("order_delivered_customer_date") == "null"
).select(
    "order_status",
    "order_delivered_customer_date",
    length("order_delivered_customer_date").alias("value_length")
).distinct().show(20, truncate=False)

Literal null text count:
+----------------------+-----------------------+
|sum(carrier_null_text)|sum(customer_null_text)|
+----------------------+-----------------------+
|                  1783|                   2965|
+----------------------+-----------------------+

Empty text count:
+------------------+-------------------+
|sum(carrier_empty)|sum(customer_empty)|
+------------------+-------------------+
|                 0|                  0|
+------------------+-------------------+

Distinct values for missing customer delivery date:
+------------+-----------------------------+------------+
|order_status|order_delivered_customer_date|value_length|
+------------+-----------------------------+------------+
|canceled    |null                         |4           |
|created     |null                         |4           |
|invoiced    |null                         |4           |
|processing  |null                         |4           |
|delivered   |null                         |4  

In [9]:
orders_silver = orders

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column_name in date_columns:
    orders_silver = orders_silver.withColumn(
        column_name,
        when(
            trim(col(column_name)).isin("null", "", "0000-00-00 00:00:00.0"),
            None
        ).otherwise(col(column_name))
    )

for column_name in date_columns:
    orders_silver = orders_silver.withColumn(
        column_name,
        to_timestamp(col(column_name), "yyyy-MM-dd HH:mm:ss.S")
    )

orders_silver.printSchema()
orders_silver.select(
    "order_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+----------------------------+-----------------------------+
|00010242fe8c5a6d1ba2dd792cb16214|delivered   |2017-09-19 18:34:16         |2017-09-20 23:43:48          |
|00018f77f2f0320c557190d7a144bdd3|delivered   |2017-05-04 14:35:00         |2017-05-12 16:04:24          |
|000229ec398224ef6ca0657da4fc7

In [10]:
orders_silver.select(
    sum(when(col("order_delivered_carrier_date").isNull(), 1).otherwise(0)).alias("carrier_date_nulls"),
    sum(when(col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("customer_date_nulls")
).show()

+------------------+-------------------+
|carrier_date_nulls|customer_date_nulls|
+------------------+-------------------+
|              1783|               2965|
+------------------+-------------------+



In [11]:
orders_silver = orders_silver.withColumn(
    "delivery_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
)

orders_silver = orders_silver.withColumn(
    "is_late",
    when(
        col("order_delivered_customer_date").isNull(),
        lit(None).cast("integer")
    ).when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
        lit(1)
    ).otherwise(lit(0))
)

orders_silver.select(
    "order_id",
    "order_status",
    "delivery_days",
    "is_late"
).show(10, truncate=False)

+--------------------------------+------------+-------------+-------+
|order_id                        |order_status|delivery_days|is_late|
+--------------------------------+------------+-------------+-------+
|00010242fe8c5a6d1ba2dd792cb16214|delivered   |7            |0      |
|00018f77f2f0320c557190d7a144bdd3|delivered   |16           |0      |
|000229ec398224ef6ca0657da4fc703e|delivered   |8            |0      |
|00024acbcdf0a6daa1e931b038114c75|delivered   |6            |0      |
|00042b26cf59d7ce69dfabb4e55b4fd9|delivered   |25           |0      |
|00048cc3ae777c65dbb7d2a0634bc1ea|delivered   |7            |0      |
|00054e8431b9d7675808bcb819fb4a32|delivered   |8            |0      |
|000576fe39319847cbb9d288c5617fa6|delivered   |5            |0      |
|0005a1a1728c9d785b8e2b08b904576c|delivered   |10           |1      |
|0005f50442cb953dcd1d21e1fb923495|delivered   |2            |0      |
+--------------------------------+------------+-------------+-------+
only showing top 10 

In [12]:
print("Delivery days statistics:")
orders_silver.select("delivery_days").summary().show()

print("Invalid negative delivery days:")
orders_silver.filter(col("delivery_days") < 0).count()

print("is_late distribution:")
orders_silver.groupBy("is_late").count().orderBy("is_late").show()

print("Missing delivery days by order status:")
orders_silver.groupBy("order_status").agg(
    sum(when(col("delivery_days").isNull(), 1).otherwise(0)).alias("missing_delivery_days"),
    sum(when(col("delivery_days").isNotNull(), 1).otherwise(0)).alias("available_delivery_days")
).orderBy("order_status").show()

Delivery days statistics:


+-------+------------------+
|summary|     delivery_days|
+-------+------------------+
|  count|             96476|
|   mean|12.497336125046644|
| stddev| 9.555459598617164|
|    min|                 0|
|    25%|                 7|
|    50%|                10|
|    75%|                16|
|    max|               210|
+-------+------------------+

Invalid negative delivery days:
is_late distribution:
+-------+-----+
|is_late|count|
+-------+-----+
|   null| 2965|
|      0|88649|
|      1| 7827|
+-------+-----+

Missing delivery days by order status:
+------------+---------------------+-----------------------+
|order_status|missing_delivery_days|available_delivery_days|
+------------+---------------------+-----------------------+
|    approved|                    2|                      0|
|    canceled|                  619|                      6|
|     created|                    5|                      0|
|   delivered|                    8|                  96470|
|    invoiced|    

In [13]:
silver_path = "/user/hadoop/commerce_silver"

orders_silver.write.mode("overwrite").parquet(
    f"{silver_path}/orders"
)

In [14]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

customers = (
    spark.read
    .option("sep", ",")
    .schema(customers_schema)
    .csv("/user/hadoop/commerce_storage/customers")
)

customers.printSchema()
customers.show(5, truncate=False)
print("Rows:", customers.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|00012a2ce6f8dcda20d059ce98491703|248ffe10d632bebe4f7267f1f44844c9|06273                   |osasco       |SP            |
|000161a058600d5901f007fab4c27140|b0015e09bb4b6e47c52844fab5fb6638|35550                   |itapecerica  |MG            |
|0001fd6190edaaf884bcaf3d49edf079|94b11d37cd61cb2994a194d11f89682b|29830                   |nova venecia |ES            |
|0002414f95344307404f0

In [15]:
print("Null values per column:")

customers.select([
    sum(
        when(
            col(c).isNull() | (col(c) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in customers.columns
]).show(truncate=False)

print("Duplicate customer_id count:")

customers.groupBy("customer_id").count().filter(
    col("count") > 1
).count()

print("Customer state distribution:")

customers.groupBy("customer_state").count().orderBy(
    col("count").desc()
).show()

Null values per column:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|0          |0                 |0                       |0            |0             |
+-----------+------------------+------------------------+-------------+--------------+

Duplicate customer_id count:


Customer state distribution:
+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



In [16]:
duplicate_customer_ids = customers.groupBy("customer_id").count().filter(
    col("count") > 1
).count()

print("Duplicate customer_id count:", duplicate_customer_ids)

customers_silver = (
    customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .withColumn(
        "customer_zip_code_prefix",
        trim(col("customer_zip_code_prefix"))
    )
)

print("Raw rows:", customers.count())
print("Silver rows:", customers_silver.count())
customers_silver.show(5, truncate=False)

Duplicate customer_id count: 0
Raw rows: 99441


Silver rows: 99441
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|01d190d14b00073f76e0a5ec46166352|2e5dcf79b225e8d1673671db21933168|02925                   |sao paulo            |SP            |
|03a7750fc7a7bfbd7a84b2f4f26b92f1|ae7e471f70f6fb521a3dc8770cefa369|83430                   |campina grande do sul|PR            |
|04495037fc6899faffa41ba3bc4272b4|c611b2ddcec5427603ec76ba4c117373|15953                   |botelho              |SP            |
|04b7d26bde4f2d2fee0043ef81e664b1|9ef6d1d9fdc6511e44eb6b7c68a765e9|38067                   |uberaba              |MG            |
|04cef6b920c0d8f16702cab269b59044|6d0b86c615a3aa7ef4a13555ea965ef7|0318

In [17]:
customers_silver.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_silver/customers"
)

In [18]:
sellers_schema = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])

sellers = (
    spark.read
    .option("sep", ",")
    .schema(sellers_schema)
    .csv("/user/hadoop/commerce_storage/sellers")
)

print("Rows:", sellers.count())
print("Columns:", len(sellers.columns))
sellers.printSchema()
sellers.show(5, truncate=False)

Rows: 3096
Columns: 4
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------------------+----------------------+-----------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city|seller_state|
+--------------------------------+----------------------+-----------+------------+
|0015a82c2db000af6aaaf3ae2ecb0532|09080                 |santo andre|SP          |
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES          |
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ          |
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP          |
|003554e2dce176b5555353e4f3555ac8|74565                 |goiania    |GO          |
+--------------------------------+----------------------+-----------+------------+
only showing top 5 rows



In [19]:
print("Rows with null or empty seller_id:")
sellers.filter(
    col("seller_id").isNull() |
    (trim(col("seller_id")) == "")
).show(10, truncate=False)

print("Number of null/empty seller_id rows:")
print(
    sellers.filter(
        col("seller_id").isNull() |
        (trim(col("seller_id")) == "")
    ).count()
)

print("Duplicate seller_id count:")
print(
    sellers.groupBy("seller_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

Rows with null or empty seller_id:
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
+---------+----------------------+-----------+------------+

Number of null/empty seller_id rows:
0
Duplicate seller_id count:


0


In [20]:
print("Distinct seller_id count:")
print(sellers.select("seller_id").distinct().count())

print("Seller IDs with unexpected format:")
sellers.filter(
    ~trim(col("seller_id")).rlike("^[0-9a-fA-F]{32}$")
).show(20, truncate=False)

print("Rows with unexpected seller_id length:")
sellers.select(
    "seller_id",
    length(trim(col("seller_id"))).alias("id_length"),
    "seller_city",
    "seller_state"
).filter(col("id_length") != 32).show(20, truncate=False)

Distinct seller_id count:


3096
Seller IDs with unexpected format:
+-------------+----------------------+-----------+------------+
|seller_id    |seller_zip_code_prefix|seller_city|seller_state|
+-------------+----------------------+-----------+------------+
|io de janeiro|RJ                    |null       |null        |
+-------------+----------------------+-----------+------------+

Rows with unexpected seller_id length:
+-------------+---------+-----------+------------+
|seller_id    |id_length|seller_city|seller_state|
+-------------+---------+-----------+------------+
|io de janeiro|13       |null       |null        |
+-------------+---------+-----------+------------+



In [21]:
sellers_silver = (
    sellers
    .filter(
        col("seller_id").isNotNull() &
        trim(col("seller_id")).rlike("^[0-9a-fA-F]{32}$")
    )
    .dropDuplicates(["seller_id"])
    .withColumn("seller_city", lower(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .withColumn(
        "seller_zip_code_prefix",
        trim(col("seller_zip_code_prefix"))
    )
)

print("Raw sellers rows:", sellers.count())
print("Invalid rows removed:", sellers.count() - sellers_silver.count())
print("Silver sellers rows:", sellers_silver.count())

sellers_silver.show(5, truncate=False)

Raw sellers rows: 3096


Invalid rows removed: 1


Silver sellers rows: 3095
+--------------------------------+----------------------+---------------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city          |seller_state|
+--------------------------------+----------------------+---------------------+------------+
|062ce95fa2ad4dfaedfc79260130565f|95913                 |lajeado              |RS          |
|0b64bcdb0784abc139af04077d49a20e|92420                 |canoas               |RS          |
|0ea22c1cfbdc755f86b9b54b39c16043|35700                 |sete lagoas          |MG          |
|2009a095de2a2a41626f6c6d7722678d|15025                 |sao jose do rio preto|SP          |
|297d5eccd19fa9a83b2630071ff105e4|80710                 |curitiba             |PR          |
+--------------------------------+----------------------+---------------------+------------+
only showing top 5 rows



In [22]:
sellers_silver.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_silver/sellers"
)

In [23]:
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

order_items = (
    spark.read
    .option("sep", ",")
    .schema(order_items_schema)
    .csv("/user/hadoop/commerce_storage/order_items")
)

print("Rows:", order_items.count())
print("Columns:", len(order_items.columns))
order_items.printSchema()
order_items.show(5, truncate=False)

Rows: 112650
Columns: 7
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+---------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date  |price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+---------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35.0|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1     

In [24]:
print("Null values per column:")

order_items.select([
    sum(
        when(
            col(c).isNull() | (col(c) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in order_items.columns
]).show(truncate=False)

print("Duplicate composite keys:")

duplicate_item_keys = (
    order_items
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate (order_id, order_item_id):", duplicate_item_keys)

print("Negative prices:", order_items.filter(col("price") < 0).count())
print("Zero prices:", order_items.filter(col("price") == 0).count())
print("Negative freight values:", order_items.filter(col("freight_value") < 0).count())

print("Price and freight statistics:")
order_items.select("price", "freight_value").summary().show()

Null values per column:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|0       |0            |0         |0        |0                  |0    |0            |
+--------+-------------+----------+---------+-------------------+-----+-------------+

Duplicate composite keys:


Duplicate (order_id, order_item_id): 0
Negative prices: 0
Zero prices: 0
Negative freight values: 0
Price and freight statistics:
+-------+------------------+------------------+
|summary|             price|     freight_value|
+-------+------------------+------------------+
|  count|            112650|            112650|
|   mean|120.65373901465038|19.990319928982803|
| stddev| 183.6339280502592|15.806405412297103|
|    min|              0.85|               0.0|
|    25%|              39.9|             13.08|
|    50%|             74.99|             16.26|
|    75%|             134.9|             21.15|
|    max|            6735.0|            409.68|
+-------+------------------+------------------+



In [25]:
order_items_silver = (
    order_items
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("seller_id", trim(col("seller_id")))
    .withColumn(
        "shipping_limit_date",
        to_timestamp(
            col("shipping_limit_date"),
            "yyyy-MM-dd HH:mm:ss.S"
        )
    )
    .withColumn(
        "total_item_value",
        col("price") + col("freight_value")
    )
)

print("Raw rows:", order_items.count())
print("Silver rows:", order_items_silver.count())

order_items_silver.select(
    "order_id",
    "order_item_id",
    "price",
    "freight_value",
    "total_item_value",
    "shipping_limit_date"
).show(10, truncate=False)

Raw rows: 112650


Silver rows: 112650


+--------------------------------+-------------+------+-------------+------------------+-------------------+
|order_id                        |order_item_id|price |freight_value|total_item_value  |shipping_limit_date|
+--------------------------------+-------------+------+-------------+------------------+-------------------+
|0103878b7ed86b6ec873cfae01379472|1            |229.99|35.37        |265.36            |2018-08-20 00:15:41|
|01be661b8196707ef60f062632d6d1bd|1            |89.9  |12.13        |102.03            |2017-05-24 10:42:27|
|01fba151a6fa8315ff6dc6f29fe2f2d4|1            |38.5  |19.88        |58.379999999999995|2017-07-14 14:23:25|
|021f21b778ea0a81da47ebe141b1b29c|1            |38.4  |12.69        |51.089999999999996|2017-11-03 16:49:35|
|021f26893462ec6a677baba7f06ce4c1|1            |49.95 |11.85        |61.800000000000004|2017-07-18 15:32:13|
|02346fd715e28d8e4fdb6893cd121f71|1            |146.99|16.66        |163.65            |2017-11-14 21:26:31|
|02dbd29bc7a490d582

In [26]:
order_items_silver = order_items_silver.withColumn(
    "total_item_value",
    round("total_item_value", 2)
)

order_items_silver.select(
    "price",
    "freight_value",
    "total_item_value"
).show(10)

+------+-------------+----------------+
| price|freight_value|total_item_value|
+------+-------------+----------------+
|229.99|        35.37|          265.36|
|  89.9|        12.13|          102.03|
|  38.5|        19.88|           58.38|
|  38.4|        12.69|           51.09|
| 49.95|        11.85|            61.8|
|146.99|        16.66|          163.65|
|  48.3|        21.15|           69.45|
|  61.5|        13.58|           75.08|
| 89.99|        13.65|          103.64|
| 85.71|        11.69|            97.4|
+------+-------------+----------------+
only showing top 10 rows



In [27]:
order_items_silver.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_silver/order_items"
)

In [28]:
silver_tables = {
    "orders_silver": "/user/hadoop/commerce_silver/orders",
    "customers_silver": "/user/hadoop/commerce_silver/customers",
    "sellers_silver": "/user/hadoop/commerce_silver/sellers",
    "order_items_silver": "/user/hadoop/commerce_silver/order_items"
}

for table_name, table_path in silver_tables.items():
    df = spark.read.parquet(table_path)
    print(table_name, "rows =", df.count(), "columns =", len(df.columns))

orders_silver rows = 99441 columns = 10


customers_silver rows = 99441 columns = 5


sellers_silver rows = 3095 columns = 4


order_items_silver rows = 112650 columns = 8


# Transformation and Data Modeling

In [29]:
customers_silver = spark.read.parquet("/user/hadoop/commerce_silver/customers")

In [30]:
customers_silver.show()

+--------------------+--------------------+------------------------+--------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------+--------------+
|0063913c2f1878cc4...|3c4abb94aa82c0be6...|                   13178|        sumare|            SP|
|00e8bdabd8d9dec77...|3fcda7fdc267ed83f...|                   36420|   ouro branco|            MG|
|02405cd33ab625b31...|9696892d2f285e3e3...|                   30140|belo horizonte|            MG|
|0263aeaed91e6e437...|71cb999c1d3226677...|                   02349|     sao paulo|            SP|
|035cbd7a946d33043...|ca994abc57b0bd798...|                   90570|  porto alegre|            RS|
|037fefe4072b01df6...|677378f2b0b40f72c...|                   24230|       niteroi|            RJ|
|0581d4ef9081dc26e...|2587940c36e40620f...|                   13930|   serra negra|            SP|
|058d7abec

In [31]:
customers_silver.dtypes

[('customer_id', 'string'),
 ('customer_unique_id', 'string'),
 ('customer_zip_code_prefix', 'string'),
 ('customer_city', 'string'),
 ('customer_state', 'string')]

In [32]:
dim_customer = customers_silver

In [33]:
dim_customer.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_gold/dim_customer"
)

In [34]:
sellers_silver = spark.read.parquet("/user/hadoop/commerce_silver/sellers")

In [35]:
sellers_silver.show()

+--------------------+----------------------+--------------------+------------+
|           seller_id|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+----------------------+--------------------+------------+
|01fd077212124329b...|                 14079|ribeirao preto / ...|          SP|
|07d75e33f2750d97d...|                 02117|           sao paulo|          SP|
|0a198e95d32b1be2d...|                 32260|            contagem|          MG|
|1e9d5a33694bddb76...|                 78820|             jaciara|          MT|
|46dc3b2cc0980fb8e...|                 22240|      rio de janeiro|          RJ|
|4917cee8d902e1342...|                 89031|            blumenau|          SC|
|4c4d546adf3c3868f...|                 87013|             maringa|          PR|
|5563732abe64c3036...|                 07192|           guarulhos|          SP|
|6dd7dce75cd55c1ce...|                 88137|             palhoca|          SC|
|6ee85be3693ed79a8...|                 0

In [36]:
dim_seller = sellers_silver

In [37]:
dim_seller.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_gold/dim_seller"
)

In [38]:
orders_silver = spark.read.parquet("/user/hadoop/commerce_silver/orders")

dates = orders_silver.select(
    min(least(col("order_purchase_timestamp"), col("order_delivered_customer_date"))).alias("start_date"),
    max(greatest(col("order_purchase_timestamp"), col("order_delivered_customer_date"))).alias("end_date")).first()

print(dates["start_date"])
print(dates["end_date"])

2016-09-04 21:15:19
2018-10-17 17:30:18


In [39]:
base_dates = spark.sql(f"""
    SELECT explode(
        sequence(to_date('{dates["start_date"]}'), to_date('{dates["end_date"]}'), interval 1 day)
    ) as full_date
""")


dim_date = base_dates.select(
    date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), #surrogate Key
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    dayofmonth("full_date").alias("day"),
    dayofweek("full_date").alias("day_of_week"), # 1 = Sunday, 7 = Saturday
    date_format("full_date", "EEEE").alias("day_name"),   
    expr("dayofweek(full_date) IN (1, 7)").alias("is_weekend")
)

#show the schema and a sample of the data
dim_date.printSchema()
dim_date.orderBy("full_date").show(5)

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = false)
 |-- year: integer (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- day: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- day_name: string (nullable = false)
 |-- is_weekend: boolean (nullable = false)

+--------+----------+----+-------+-----+----------+---+-----------+---------+----------+
|date_key| full_date|year|quarter|month|month_name|day|day_of_week| day_name|is_weekend|
+--------+----------+----+-------+-----+----------+---+-----------+---------+----------+
|20160904|2016-09-04|2016|      3|    9| September|  4|          1|   Sunday|      true|
|20160905|2016-09-05|2016|      3|    9| September|  5|          2|   Monday|     false|
|20160906|2016-09-06|2016|      3|    9| September|  6|          3|  Tuesday|     false|
|20160907|2016-09-07|2016|      3|    9| September|

In [40]:
dim_date.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_gold/dim_date"
)

In [41]:
orders_silver.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- is_late: integer (nullable = true)



In [42]:
order_items_silver = spark.read.parquet("/user/hadoop/commerce_silver/order_items")

order_items_silver.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)



In [43]:
aggregated_orders = orders_silver.join(order_items_silver, on="order_id", how="inner")\
.filter((col("order_status") == "delivered") & col("order_delivered_customer_date").isNotNull())\
.groupBy("order_id", "customer_id", "order_purchase_timestamp", "order_approved_at",
         "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
         "delivery_days", "is_late")\
.agg(sum("freight_value").alias("shipping_cost"), 
    sum("price").alias("total_order_price"),
    count("order_item_id").alias("items_count"),
    expr("max_by(seller_id, freight_value)").alias("seller_id"))

fact_logistics = aggregated_orders.join(dim_customer, on="customer_id", how="inner")\
.join(dim_seller, on="seller_id", how="inner")\
.select(
    "order_id",
    col("customer_id"),
    col("seller_id"),
    date_format("order_purchase_timestamp", "yyyyMMdd").cast("int").alias("order_date_key"),
    date_format("order_delivered_customer_date", "yyyyMMdd").cast("int").alias("delivery_date_key"),
    datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")).alias("delivery_time_delta_days"),
    col("is_late").alias("is_late_delivery"),
    col("delivery_days").alias("order_to_delivery_days"),
    datediff(col("order_delivered_carrier_date"), col("order_approved_at")).alias("approval_to_ship_days"),
    "shipping_cost",
    "total_order_price",
    "items_count"
)

In [45]:
fact_logistics.write.mode("overwrite").parquet(
    "/user/hadoop/commerce_gold/fact_logistics"
)

In [46]:
# csv version
fact_logistics.coalesce(1).write.mode("overwrite").option("header", "true").csv("/user/hadoop/commerce_gold_csv/fact_logistics")
dim_customer.coalesce(1).write.mode("overwrite").option("header", "true").csv("/user/hadoop/commerce_gold_csv/dim_customer")
dim_seller.coalesce(1).write.mode("overwrite").option("header", "true").csv("/user/hadoop/commerce_gold_csv/dim_seller")
dim_date.coalesce(1).write.mode("overwrite").option("header", "true").csv("/user/hadoop/commerce_gold_csv/dim_date")